# Lesson 11 — Evaluation

**You will:** read a `Report` to understand what an agent did, then write tests for an agent.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/11-evaluation/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

You cannot improve what you cannot measure. Every BareBear run produces a **`Report`** — a structured trace of every step, every tool call, every token, every dollar. The report is not optional debug logging; it is *the output* of running an agent. The final answer is just one field on it.

Reports unlock three things:

1. **Audit** — exactly what tools did the agent call, with what arguments, in what order?
2. **Cost analysis** — token counts and dollar costs are tracked, not estimated.
3. **Testing** — you can write assertions over reports.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")
print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")

## A small test suite for an email-reply agent

In [ ]:
from barebear import Bear, Task, Tool, Policy, OpenRouterModel

def read_ticket(ticket_id: str) -> str:
    return f"Ticket {ticket_id}: customer locked out."

def draft_email(to: str, subject: str, body: str) -> str:
    return f"DRAFT to {to}: {subject}"

def send_email(to: str, subject: str, body: str) -> str:
    return f"Email sent to {to}"


def build_email_agent():
    return Bear(
        model=OpenRouterModel(),
        tools=[
            Tool(name="read_ticket", fn=read_ticket, description="Read a ticket"),
            Tool(name="draft_email", fn=draft_email, description="Draft an email"),
            Tool(name="send_email", fn=send_email, description="Send an email",
                 requires_approval=True, side_effects="external"),
        ],
        policy=Policy(require_approval_for=["send_email"], max_cost_usd=0.05),
    )


def test_agent_drafts_before_sending():
    bear = build_email_agent()
    report = bear.run(Task(goal="Reply to ticket TK-42"))

    tool_steps = [s for s in report.steps if s["type"] == "tool_call"]
    tool_names = [s["tool_name"] for s in tool_steps]

    assert "read_ticket" in tool_names, "should read the ticket"
    assert "draft_email" in tool_names, "should draft before pause"
    print("PASS: agent drafted before sending")


def test_agent_pauses_for_send_approval():
    bear = build_email_agent()
    report = bear.run(Task(goal="Reply to ticket TK-42"))
    assert report.status == "paused", f"expected paused, got {report.status}"
    assert report.checkpoint_id is not None
    print("PASS: agent paused for approval")


test_agent_drafts_before_sending()
test_agent_pauses_for_send_approval()

## What's interesting here

These assertions pin down the **behaviour** — the pattern of tool calls, the safety pause — independent of whatever text the model happened to produce that day. That's the only kind of test that survives across model versions.

## Exercise

1. Add a budget assertion: `assert report.total_cost_usd < 0.02`. Find a sensible threshold by running the test three times.
2. Two prompts, one task: which costs less, and which is faster? Build a tiny eval harness that runs both five times and reports averages.

## What's next

[Lesson 12 →](../12-honest-uncertainty/lesson.ipynb)